In [72]:
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor

In [2]:
import sklearn
np.__version__, pd.__version__, sklearn.__version__

('2.4.3', '3.0.1', '1.8.0')

## Set 1

In [ ]:
df = pd.read_csv("set_01_data.csv")
df.head(2)

In [ ]:
df_q1 = df[["Picture_quality", "Speaker", "Frequency"]].copy()
df_q1.head(2)

In [ ]:
# ser_u = df_q1["Picture_quality"].drop_duplicates()
# ser_u = df_q1["Speaker"].drop_duplicates()
ser_u = df_q1["Frequency"].drop_duplicates()
# ser_u[ser_u.str.contains("Hz")]
ser_u[ser_u.str.contains("[0-9]{2,3} Hz")] # 숫자 2~3자리 + 한 칸 띄어쓰기 + "Hz"

In [14]:
df_q1["hz1"] = df_q1["Picture_quality"].str.contains("Hz") + 0
df_q1["hz2"] = df_q1["Speaker"        ].str.contains("Hz") + 0
df_q1["hz3"] = df_q1["Frequency"      ].str.contains("Hz") + 0
df_q1["hz_cnt"] = df_q1["hz1"] + df_q1["hz2"] + df_q1["hz3"]

In [17]:
df_q1["hz_cnt"].value_counts()

hz_cnt
1    659
0      7
Name: count, dtype: int64

In [19]:
df_q1_sub = df_q1.loc[df_q1["hz_cnt"] > 0, ]
len(df_q1_sub)

659

In [ ]:
ser_cnt = df_q1_sub.loc[:, :"Frequency"].apply(lambda x: x.str.contains("60 Hz").sum(), axis = 1)
ser_cnt.head()

In [24]:
ser_cnt.value_counts() # 493!!!

1    493
0    166
Name: count, dtype: int64

### Q2.

In [28]:
df = pd.read_csv("set_01_data.csv")

In [31]:
con_1 = df["Stars"  ] != 0
con_2 = df["Ratings"] != 0
con_3 = df["Reviews"] != 0
df_sub = df.loc[con_1 & con_2 & con_3, ]
len(df_sub)

225

In [39]:
def check_4k_counter(x):
    return x.str.contains("4K").sum()

ls_cols = ["channel", "Operating_system", "Picture_quality"]
# ser_4k = df_sub[ls_cols].apply(check_4k_counter, axis = 1)
ser_4k = df_sub[ls_cols].apply(lambda x: x.str.contains("4K").sum(), axis = 1)
ser_4k.value_counts()

1    113
0    112
Name: count, dtype: int64

In [41]:
df_sub = df_sub.loc[ser_4k >= 1, ].reset_index(drop = True)
len(df_sub)

113

In [79]:
# df_model = df_sub.loc[:, df_sub.dtypes != "object"] # 시험버전
df_model = df_sub.loc[:, df_sub.dtypes != "str"] # 최신버전
df_model.head(2)

,Stars,Ratings,Reviews,current_price,MRP
0,4.4,9687,1532,62999,79999
1,4.5,17462,5306,54999,69999


In [47]:
# df_sub.dtypes

In [ ]:
model_kmeans = KMeans(n_clusters = 3, random_state = 123)
model_kmeans.fit(df_model)

In [54]:
model_kmeans.labels_[:4]

array([0, 0, 2, 0], dtype=int32)

In [52]:
df_model["cluster"] = model_kmeans.labels_

In [ ]:
df_model.head(2)

In [55]:
df_model["cluster"].value_counts() # 55!!

cluster
2    55
0    44
1    14
Name: count, dtype: int64

### Q3.

In [ ]:
df = pd.read_csv("set_01_data.csv")
df.head(1)

In [59]:
df_q3 = df.loc[~df["channel"].str.contains("Pixel|Oper"), ].reset_index(drop = True)
len(df_q3)

548

In [64]:
df_q3["x1"] = df_q3["Reviews"] / df_q3["Ratings"]
df_q3["x2"] = df_q3["MRP"]
df_q3["x3"] = df_q3["current_price"] / df_q3["MRP"]
df_q3["x4"] = df_q3["channel"].str.contains("Netflix") + 0
df_q3["x5"] = df_q3["channel"].str.contains("Prime Video") + 0
df_q3["x6"] = df_q3["Picture_quality"].str.contains("4K|8K") + 0 # 대소문자 주의!!!

In [ ]:
df_q3[["x4", "x5", "x6"]].sum() # 오타가 난 경우 여기에서 0이 보일 수 있음!!🚨🚨🚨

In [66]:
# 머신러닝 모델링 시 학습/평가 데이터 세트의 데이터프레임 객체에
# 종속변수는 가장 왼쪽(앞)에 위치시키는 것이 코드 작성에 유리.
df_model = df_q3[["Stars", "x1", "x2", "x3", "x4", "x5", "x6"]].copy()
df_model.head(2)

,Stars,x1,x2,x3,x4,x5,x6
0,4.3,NaN,89990,0.444383,1,1,1
1,0.0,NaN,45900,0.997495,1,0,0


In [ ]:
df_model.isna().sum()

In [68]:
len(df_model)

548

In [69]:
df_model = df_model.dropna()

In [70]:
len(df_model)

200

In [ ]:
df_model.head(2)

In [ ]:
model_rf = RandomForestRegressor(random_state = 123)
model_rf.fit(X = df_model.drop(columns = "Stars"),
             y = df_model["Stars"])

In [ ]:
arr_MDI = model_rf.feature_importances_
arr_MDI

In [77]:
ser_imp = pd.Series(arr_MDI, index = df_model.columns[1:])
ser_imp # x3 - 할인율!!

x1    0.275090
x2    0.197035
x3    0.466081
x4    0.023827
x5    0.024440
x6    0.013527
dtype: float64